# 미니 GPT 구현
LLM 스터디에서 배운 토큰화, 임베딩, Self-Attention, 트랜스포머, 훈련 루프를 한 파일로 구현함.
글자 단위 토크나이저를 쓰는 소형 모델이며, 구조는 GPT와 같음.

### 0. 준비: 라이브러리와 학습 텍스트
PyTorch를 불러오고 학습 텍스트를 읽음. 텍스트는 한국어 웹 글 약 3천만 자이며 `prepare_data.py`로 생성함.
`mini_gpt_mps.ipynb`와 같은 파일을 씀.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)  # 재실행해도 같은 결과가 나오게 seed 고정

# ── 0. 학습용 텍스트: data/input.txt (없으면 python prepare_data.py 먼저 실행) ──
text = open("data/input.txt", encoding="utf-8").read().strip()

### 1. 토큰화 (2-2, 2-3)
텍스트를 숫자(토큰 ID)로 바꾸는 단계임.
글자 하나를 토큰 하나로 처리하며, `stoi`(어휘사전)와 `itos`(역어휘사전)로 글자와 ID를 서로 변환함.

In [2]:
# ── 2-2, 2-3. 토큰화: 여기선 "글자 하나 = 토큰 하나"로 단순화 ──
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}   # 문자 -> ID (어휘사전)
itos = {i: ch for ch, i in stoi.items()}       # ID -> 문자 (역어휘사전)

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)

### 2. 슬라이딩 윈도로 입력-타깃 쌍 만들기 (2-6)
텍스트에서 입력(x)과 한 칸 뒤로 민 정답(y)을 잘라 만듦. 시작 위치는 무작위로 뽑음.
앞 글자들을 보고 다음 글자를 맞히는 것이 학습 목표임.

In [3]:
# ── 하이퍼파라미터 (모델 크기: 일부러 아주 작게) ──
context_length = 16   # 한 번에 보는 토큰 개수 (max_length)
embed_dim = 32        # 임베딩 차원 (GPT-2는 768, 여기선 작게 잡음)
n_heads = 4
n_layers = 2
lr = 1e-3
epochs = 500

# ── 2-6. 슬라이딩 윈도로 입력-타깃 쌍 만들기 ──
def get_batch(batch_size=8):
    ix = torch.randint(0, len(data) - context_length - 1, (batch_size,))
    x = torch.stack([data[i:i + context_length] for i in ix])
    y = torch.stack([data[i + 1:i + context_length + 1] for i in ix])  # 타깃 = 입력을 한 칸 민 것
    return x, y

# 확인: 타깃은 입력을 한 칸 민 것임임
x, y = get_batch(1)
print('입력:', decode(x[0].tolist()))
print('정답:', decode(y[0].tolist()))

입력:  해야 한다는 입장을 고수하기
정답: 해야 한다는 입장을 고수하기도


### 3. Self-Attention
현재 토큰이 앞의 토큰 중 어디를 얼마나 참고할지를 Q, K, V로 계산함.
미래 토큰은 마스크로 가려서 다음 글자 예측 시 정답을 볼 수 없게 함(인과적 마스킹).

In [4]:
# ── 새로 추가: Self-Attention (다음 장 내용 최소 버전) ──
# "지금 토큰이 앞의 토큰들 중 어디를 더 주의 깊게 볼지" 가중치를 계산하는 블록
class SelfAttention(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)  # Query, Key, Value를 한 번에 생성
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, C = x.shape  # 배치, 시퀀스길이, 임베딩차원
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)  # 각각 (B, T, n_heads, head_dim)
        q, k, v = [t.transpose(1, 2) for t in (q, k, v)]  # (B, n_heads, T, head_dim)

        # Q와 K를 곱해서 "얼마나 관련 있는지" 점수 계산 (A-2의 .matmul())
        att = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        # 인과적 마스킹: 미래 토큰은 못 보게 막음 (다음 단어 예측이니 당연히 필요)
        mask = torch.tril(torch.ones(T, T)).bool()
        att = att.masked_fill(~mask, float("-inf"))
        att = F.softmax(att, dim=-1)  # 점수를 확률처럼 정규화 (앞서 배운 softmax)

        out = att @ v  # 가중합으로 최종 벡터 생성
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.proj(out)

### 4. 트랜스포머 블록과 전체 모델 (A-5, 2-7, 2-8)
`nn.Module`을 상속해 어텐션, 피드포워드, 잔차 연결을 블록으로 묶음(A-5).
토큰 임베딩(2-7)에 위치 임베딩(2-8)을 더한 값을 입력으로 씀.

In [5]:
# ── A-5 스타일: nn.Module 상속해서 트랜스포머 블록 하나 구성 ──
class Block(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = SelfAttention(embed_dim, n_heads)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # 잔차 연결 (원본 + 어텐션 결과)
        x = x + self.ff(self.ln2(x))     # 잔차 연결 (원본 + 피드포워드 결과)
        return x

# ── 2-7, 2-8. 토큰 임베딩 + 위치 임베딩을 포함한 전체 모델 ──
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, context_length, n_heads, n_layers):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)      # 2-7
        self.pos_emb = nn.Embedding(context_length, embed_dim)    # 2-8
        self.blocks = nn.Sequential(*[Block(embed_dim, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size)  # 최종적으로 "다음 토큰 확률"로 변환

    def forward(self, x, targets=None):
        B, T = x.shape
        tok = self.token_emb(x)                                  # (B, T, embed_dim)
        pos = self.pos_emb(torch.arange(T))                      # (T, embed_dim)
        h = tok + pos                                             # 2-8: 토큰임베딩 + 위치임베딩
        h = self.blocks(h)
        h = self.ln_f(h)
        logits = self.head(h)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, start_text, max_new_tokens=50):
        self.eval()
        ids = torch.tensor([encode(start_text)], dtype=torch.long)
        for _ in range(max_new_tokens):
            ids_cond = ids[:, -context_length:]
            logits, _ = self(ids_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)  # 마지막 위치의 다음글자 확률
            next_id = torch.multinomial(probs, num_samples=1)
            ids = torch.cat([ids, next_id], dim=1)
        self.train()
        return decode(ids[0].tolist())

### 5. 훈련 전 생성
모델을 만들고 훈련 전 텍스트를 먼저 생성해 둠. 가중치가 무작위라 의미 없는 글자가 나옴. 훈련 후와 비교하는 기준임.

In [6]:
# ── A-7 스타일: 일반적인 훈련 루프 ──
model = MiniGPT(vocab_size, embed_dim, context_length, n_heads, n_layers)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

print(f"어휘 크기: {vocab_size}, 파라미터 수: {sum(p.numel() for p in model.parameters())}")
print("── 훈련 전 생성 결과 (랜덤 초기 가중치라 의미 없음) ──")
before_text = model.generate("지원은 ", max_new_tokens=30)
print(before_text)

어휘 크기: 1473, 파라미터 수: 121729
── 훈련 전 생성 결과 (랜덤 초기 가중치라 의미 없음) ──
지원은 ×략않봅낸본웬봇강녹대깜륨기뜬닫X공닛쏙험平펀벚남열끈벤캔렐


### 6. 훈련 루프 (A-7)
배치 추출, 예측, loss 계산, 역전파, 가중치 갱신을 500번 반복함.
loss는 정답과 예측의 차이이며, 내려가면 학습이 진행되는 것임.

In [7]:
loss_history = []

for epoch in range(epochs):
    x, y = get_batch()
    logits, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    if epoch % 100 == 0:
        print(f"epoch {epoch}: loss {loss.item():.4f}")

epoch 0: loss 7.4210


epoch 100: loss 4.9108


epoch 200: loss 5.1796


epoch 300: loss 4.7082


epoch 400: loss 4.4590


### 7. 훈련 후 생성
같은 프롬프트 `"지원은 "`로 다시 생성함.

In [8]:
after_text = model.generate("지원은 ", max_new_tokens=50)
print(after_text)

지원은 덕정고 눈인 했에 수제악 있에 장어 컬전열기당하이렸l용라는 쉬을져는 심재공부o험는 시를 리


### 8. 훈련 전후 비교
loss는 처음 10스텝 평균과 마지막 10스텝 평균을 비교함. 생성 결과는 같은 프롬프트로 나란히 출력함.

In [9]:
from IPython.display import HTML, display
import html

first = sum(loss_history[:10]) / 10
last = sum(loss_history[-10:]) / 10
n = len(before_text)  # 훈련 전 결과와 같은 길이로 맞춤
cell = lambda s: html.escape(s).replace("\n", "<br>")

display(HTML(f"""
<table style="font-size:16px; border-collapse:collapse; width:100%">
<tr>
  <th style="width:50%; padding:8px; border:1px solid #888">훈련 전 (loss {first:.3f})</th>
  <th style="width:50%; padding:8px; border:1px solid #888">훈련 후 (loss {last:.3f})</th>
</tr>
<tr>
  <td style="padding:12px; border:1px solid #888; vertical-align:top">{cell(before_text)}</td>
  <td style="padding:12px; border:1px solid #888; vertical-align:top">{cell(after_text[:n])}</td>
</tr>
</table>
"""))
print(f"loss: {first:.3f} → {last:.3f}  ({(1 - last / first) * 100:.0f}% 감소)")

훈련 전 (loss 7.324),훈련 후 (loss 4.613)
지원은 ×략않봅낸본웬봇강녹대깜륨기뜬닫X공닛쏙험平펀벚남열끈벤캔렐,지원은 덕정고 눈인 했에 수제악 있에 장어 컬전열기당하이렸l용


loss: 7.324 → 4.613  (37% 감소)
